# Pydantic y Structured Output en ARCA


El LLM está **simulado** (mock): no hace falta GPU ni API key. Donde iría el modelo real hay una plantilla lista para cambiar.

**Qué vas a ver**

1. Qué es Pydantic.
2. Por qué la API de ARCA usa Pydantic.
3. Qué es Structured Output.
4. La clave: `description` es prompt.
5. Plantilla completa: modelo de datos → prompt → LLM → objeto validado.
6. Dónde se usa en cada fase de la práctica.
7. Errores típicos.
8. Links.

**Cómo abrirlo** (desde la raíz del repo):

```bash
uv run --with jupyter jupyter lab docs/pydantic_structured_output.ipynb
```

En VS Code: elige el kernel de `.venv` (te pedirá instalar `ipykernel`).

In [1]:
import json
import sys
import warnings
from pathlib import Path

import pydantic

warnings.filterwarnings("ignore", message="IProgress not found")  # aviso de tqdm, no afecta

# Raíz del repo = carpeta con pyproject.toml. Así funciona `from api.schemas import ...`.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ))


def ver(obj) -> None:
    """Imprime dicts y listas como JSON bonito."""
    print(json.dumps(obj, indent=2, ensure_ascii=False, default=str))


print("pydantic", pydantic.VERSION, "| repo:", RAIZ.name)

pydantic 2.13.5 | repo: dgm-arca


## 1. Qué es Pydantic

- Librería de Python para **validar datos con tipos**.
- Defines una clase con campos y tipos. Pydantic comprueba los datos al crear el objeto.
- Dato bueno → objeto. Dato malo → `ValidationError` con la lista de fallos: qué campo, qué pasó.
- Convierte cuando tiene sentido: `"39.90"` → `39.9`. Si no puede convertir, falla.
- Saca el **JSON Schema** del modelo. Esto es lo que importa para los LLM (sección 4).

Docs: [Modelos](https://pydantic.dev/docs/validation/latest/concepts/models/) · [Campos (`Field`)](https://pydantic.dev/docs/validation/latest/concepts/fields/) · [JSON Schema](https://pydantic.dev/docs/validation/latest/concepts/json_schema/) · [Validadores](https://pydantic.dev/docs/validation/latest/concepts/validators/) · [Errores](https://pydantic.dev/docs/validation/latest/errors/validation_errors/)

In [2]:
from typing import Literal

from pydantic import BaseModel, Field, ValidationError


class Producto(BaseModel):
    nombre: str
    precio_eur: float = Field(gt=0)
    categoria: Literal["mueble", "iluminacion", "textil"]
    etiquetas: list[str] = []


p = Producto(nombre="Lámpara Nube", precio_eur="39.90", categoria="iluminacion")
print(p)
print(type(p.precio_eur))  # entró texto, sale float

nombre='Lámpara Nube' precio_eur=39.9 categoria='iluminacion' etiquetas=[]
<class 'float'>


Dato malo. Dos fallos a la vez: precio negativo y categoría que no existe. Pydantic los da todos juntos, no para en el primero.

In [3]:
try:
    Producto(nombre="Silla", precio_eur=-5, categoria="cocina")
except ValidationError as e:
    print(e.error_count(), "errores\n")
    for err in e.errors(include_url=False):
        print(err["loc"], "->", err["msg"])

2 errores

('precio_eur',) -> Input should be greater than 0
('categoria',) -> Input should be 'mueble', 'iluminacion' or 'textil'


Ida y vuelta con JSON. Estos cuatro métodos son el 90 % del uso:

| Método | Qué hace |
|---|---|
| `Modelo.model_validate(dict)` | dict → objeto validado |
| `Modelo.model_validate_json(texto)` | texto JSON → objeto validado (esto usarás con el LLM) |
| `obj.model_dump()` / `obj.model_dump_json()` | objeto → dict / texto JSON |
| `Modelo.model_json_schema()` | modelo → JSON Schema (esto le das al LLM) |

In [4]:
texto = '{"nombre": "Cojín", "precio_eur": 12, "categoria": "textil"}'

p = Producto.model_validate_json(texto)
print(p.model_dump())
print(p.model_dump_json())
print()
ver(Producto.model_json_schema())

{'nombre': 'Cojín', 'precio_eur': 12.0, 'categoria': 'textil', 'etiquetas': []}
{"nombre":"Cojín","precio_eur":12.0,"categoria":"textil","etiquetas":[]}

{
  "properties": {
    "nombre": {
      "title": "Nombre",
      "type": "string"
    },
    "precio_eur": {
      "exclusiveMinimum": 0,
      "title": "Precio Eur",
      "type": "number"
    },
    "categoria": {
      "enum": [
        "mueble",
        "iluminacion",
        "textil"
      ],
      "title": "Categoria",
      "type": "string"
    },
    "etiquetas": {
      "default": [],
      "items": {
        "type": "string"
      },
      "title": "Etiquetas",
      "type": "array"
    }
  },
  "required": [
    "nombre",
    "precio_eur",
    "categoria"
  ],
  "title": "Producto",
  "type": "object"
}


## 2. Por qué Pydantic en una API

**Problema.** La API recibe JSON de fuera. Puede llegar cualquier cosa: campo vacío, número como texto, campo que no existe. Sin validar, el código revienta lejos del origen y el error no dice nada útil.

**Con Pydantic + FastAPI:**

- FastAPI lee el cuerpo y lo valida contra el modelo. Si está mal → **HTTP 422** con el campo que falla. Tu función ni se ejecuta.
- La salida también se valida (`response_model=`). Si tu código devuelve algo roto, el fallo sale en tu lado, no en el corrector.
- El esquema genera la documentación de `/docs` sola (OpenAPI).
- **Contrato.** El script de corrección y vosotros usáis los mismos modelos: [`api/schemas.py`](../api/schemas.py). No renombréis ni quitéis campos. Añadir campos opcionales sí se puede.

Docs: [FastAPI: cuerpo de la petición](https://fastapi.tiangolo.com/tutorial/body/) · [FastAPI: `response_model`](https://fastapi.tiangolo.com/tutorial/response-model/) · [FastAPI: tests con `TestClient`](https://fastapi.tiangolo.com/tutorial/testing/) · [`api/README.md`](../api/README.md)

In [5]:
from api.schemas import ReasoningRequest, ReasoningResponse

ver(ReasoningRequest.model_json_schema())

{
  "properties": {
    "question": {
      "description": "Problem statement for the model.",
      "minLength": 1,
      "title": "Question",
      "type": "string"
    },
    "expected_answer": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "description": "Optional ground truth; when given, the response includes the verifier verdict",
      "title": "Expected Answer"
    },
    "max_new_tokens": {
      "default": 1024,
      "maximum": 8192,
      "minimum": 16,
      "title": "Max New Tokens",
      "type": "integer"
    }
  },
  "required": [
    "question"
  ],
  "title": "ReasoningRequest",
  "type": "object"
}


In [6]:
cuerpos = [
    {"question": "¿Cuánto es 2+2?"},
    {"question": ""},  # min_length=1
    {"question": "2+2", "max_new_tokens": 99999},  # le=8192
]
for cuerpo in cuerpos:
    try:
        print("OK  ", ReasoningRequest.model_validate(cuerpo))
    except ValidationError as e:
        print("MAL ", [(err["loc"], err["msg"]) for err in e.errors()])

OK   question='¿Cuánto es 2+2?' expected_answer=None max_new_tokens=1024
MAL  [(('question',), 'String should have at least 1 character')]
MAL  [(('max_new_tokens',), 'Input should be less than or equal to 8192')]


Lo mismo, pero a través de la API real del repo. `TestClient` llama a la app sin levantar servidor.

- Cuerpo malo → **422**: lo para Pydantic.
- Cuerpo bueno → **501** si la fase 1 aún no está conectada: la validación pasó, falta el backend.

In [7]:
import logging

from fastapi.testclient import TestClient

from api.app import app

logging.disable(logging.INFO)  # silencia los logs de cada petición
cliente = TestClient(app)

r = cliente.post("/reasoning", json={"question": ""})
print(r.status_code)
ver(r.json())

r = cliente.post("/reasoning", json={"question": "¿Cuánto es 2+2?"})
print("\ncuerpo bueno ->", r.status_code)

422
{
  "detail": [
    {
      "type": "string_too_short",
      "loc": [
        "body",
        "question"
      ],
      "msg": "String should have at least 1 character",
      "input": "",
      "ctx": {
        "min_length": 1
      }
    }
  ]
}



cuerpo bueno -> 501


La salida también tiene contrato. Si tu backend olvida un campo obligatorio, revienta aquí, antes de llegar al corrector.

In [8]:
try:
    ReasoningResponse(thinking="2 y 2 son 4", answer="4")  # faltan raw y has_valid_format
except ValidationError as e:
    print([err["loc"] for err in e.errors()])

[('raw',), ('has_valid_format',)]


## 3. Qué es Structured Output

El LLM saca **texto**. Tu programa necesita **datos**: campos, tipos, listas. Structured Output = conseguir que la salida del modelo cumpla un esquema JSON.

Tres niveles, de menos a más garantía:

| Nivel | Cómo | Qué garantiza |
|---|---|---|
| 1. Pedirlo en el prompt | "Responde solo JSON con este esquema" + parsear + reintentar | Nada. Tú validas. |
| 2. JSON mode | El proveedor fuerza JSON sintácticamente válido | JSON que parsea. No tus campos. |
| 3. Decodificación restringida | El motor solo deja generar tokens que cumplen el esquema | La forma. El contenido no. |

**Cómo funciona el nivel 3.** El esquema se compila a una gramática. En cada paso de generación, el motor pone a `-inf` los logits de los tokens que romperían la gramática. El modelo solo puede elegir entre tokens válidos. Lo hacen [vLLM](https://docs.vllm.ai/en/latest/features/structured_outputs/), [Outlines](https://github.com/dottxt-ai/outlines), [XGrammar](https://github.com/mlc-ai/xgrammar), [Ollama](https://ollama.com/blog/structured-outputs), [OpenAI](https://developers.openai.com/api/docs/guides/structured-outputs) y [Claude](https://platform.claude.com/docs/en/build-with-claude/structured-outputs). Paper de base: [Willard & Louf, 2023](https://arxiv.org/abs/2307.09702).

**Dónde entra Pydantic.** En los dos extremos:

```
Modelo Pydantic ──model_json_schema()──► JSON Schema ──► prompt o response_format ──► LLM
                                                                                        │
objeto Python ◄──model_validate_json()── texto JSON ◄───────────────────────────────────┘
```

**Valida siempre**, aunque uses el nivel 3. Forma correcta no es contenido correcto. Y algunos motores no aplican todas las restricciones (mínimos, longitudes, patrones). Pydantic sí las comprueba.

## 4. La clave: `description` es prompt

El LLM **no ve tu código Python**. Ve el JSON Schema. Dentro del esquema viaja:

- el **nombre** del campo (`ancho_cm` dice más que `w`),
- el **tipo**,
- la **`description`** de cada `Field`,
- el **docstring** de la clase (sale como `description` del esquema),
- los **`examples`**,
- los valores permitidos de un `Literal` (salen como `enum`).

El modelo lee todo eso como instrucciones. **Cada `description` es un mini prompt para ese campo.** Escríbela como una instrucción a un becario: qué poner, en qué unidad, en qué formato, qué hacer si falta el dato.

Ejemplo: extraer una ficha de producto del texto libre de un proveedor (es uno de los temas de [`docs/temas_ejemplo.md`](temas_ejemplo.md)). Mismo dato, dos esquemas.

In [9]:
class FichaPobre(BaseModel):
    nombre: str
    ancho: float
    material: str
    precio: float | None
    compatible: list[str]


class FichaProducto(BaseModel):
    """Ficha normalizada de un producto, extraída del texto libre de un proveedor."""

    nombre: str = Field(
        description="Nombre comercial tal como aparece en el texto, sin la referencia SKU."
    )
    ancho_cm: float = Field(
        gt=0,
        description=(
            "Ancho en centímetros, solo el número. Si el texto da mm o pulgadas, convierte a cm."
        ),
    )
    material: Literal["madera", "metal", "plastico", "vidrio", "textil", "otro"] = Field(
        description=(
            "Material principal. Si hay varios, el de la superficie más grande. "
            "Si no encaja, 'otro'."
        )
    )
    precio_eur: float | None = Field(
        default=None,
        ge=0,
        description="Precio en euros con IVA. null si el texto no da precio. No lo inventes.",
    )
    compatible_con: list[str] = Field(
        default_factory=list,
        description=(
            "SKUs de otros productos que el texto cita como compatibles. "
            "Lista vacía si no cita ninguno."
        ),
        examples=[["SKU-1042", "SKU-2210"]],
    )


print("=== Esquema pobre ===")
ver(FichaPobre.model_json_schema())

=== Esquema pobre ===
{
  "properties": {
    "nombre": {
      "title": "Nombre",
      "type": "string"
    },
    "ancho": {
      "title": "Ancho",
      "type": "number"
    },
    "material": {
      "title": "Material",
      "type": "string"
    },
    "precio": {
      "anyOf": [
        {
          "type": "number"
        },
        {
          "type": "null"
        }
      ],
      "title": "Precio"
    },
    "compatible": {
      "items": {
        "type": "string"
      },
      "title": "Compatible",
      "type": "array"
    }
  },
  "required": [
    "nombre",
    "ancho",
    "material",
    "precio",
    "compatible"
  ],
  "title": "FichaPobre",
  "type": "object"
}


In [10]:
print("=== Esquema con descripciones ===")
ver(FichaProducto.model_json_schema())

=== Esquema con descripciones ===
{
  "description": "Ficha normalizada de un producto, extraída del texto libre de un proveedor.",
  "properties": {
    "nombre": {
      "description": "Nombre comercial tal como aparece en el texto, sin la referencia SKU.",
      "title": "Nombre",
      "type": "string"
    },
    "ancho_cm": {
      "description": "Ancho en centímetros, solo el número. Si el texto da mm o pulgadas, convierte a cm.",
      "exclusiveMinimum": 0,
      "title": "Ancho Cm",
      "type": "number"
    },
    "material": {
      "description": "Material principal. Si hay varios, el de la superficie más grande. Si no encaja, 'otro'.",
      "enum": [
        "madera",
        "metal",
        "plastico",
        "vidrio",
        "textil",
        "otro"
      ],
      "title": "Material",
      "type": "string"
    },
    "precio_eur": {
      "anyOf": [
        {
          "minimum": 0,
          "type": "number"
        },
        {
          "type": "null"
        }


**Compara.** Con el esquema pobre el modelo adivina: ¿ancho en cm o en mm? ¿material en texto libre? ¿precio inventado si no aparece? ¿qué es `compatible`? Con el esquema bueno, la respuesta a cada duda está dentro del esquema.

**Reglas para escribir `description`:**

- **Unidad y formato explícitos.** "en cm", "fecha ISO 8601, AAAA-MM-DD".
- **Qué hacer si falta el dato.** `null`, lista vacía. Esto corta alucinaciones.
- **Conjuntos cerrados con `Literal`**, no con texto en la descripción. `Literal` sale como `enum` y el motor del nivel 3 lo fuerza.
- **Corta.** Una o dos frases. Es prompt: cada token cuesta y distrae.
- **Nombre de campo claro.** El nombre también es prompt.
- **El orden importa.** El modelo genera los campos en orden. Si quieres que piense antes de responder, pon `razonamiento` antes de `respuesta` (sección 6.3).
- **`gt`, `max_length`, `pattern`** van al esquema, pero no todos los motores los aplican. Pydantic los comprueba después igualmente.

Así se mete el esquema en el prompt de sistema (nivel 1). En el nivel 3 el esquema va además en un parámetro aparte (`response_format`), pero las descripciones siguen siendo lo que el modelo lee.

In [11]:
def prompt_de_sistema(modelo: type[BaseModel]) -> str:
    esquema = json.dumps(modelo.model_json_schema(), ensure_ascii=False, indent=2)
    return (
        "Eres un extractor de datos. Lee el texto del usuario y devuelve SOLO un objeto JSON "
        "que cumpla este JSON Schema. Sin texto antes ni después.\n\n"
        f"JSON Schema:\n{esquema}"
    )


print(prompt_de_sistema(FichaProducto))

Eres un extractor de datos. Lee el texto del usuario y devuelve SOLO un objeto JSON que cumpla este JSON Schema. Sin texto antes ni después.

JSON Schema:
{
  "description": "Ficha normalizada de un producto, extraída del texto libre de un proveedor.",
  "properties": {
    "nombre": {
      "description": "Nombre comercial tal como aparece en el texto, sin la referencia SKU.",
      "title": "Nombre",
      "type": "string"
    },
    "ancho_cm": {
      "description": "Ancho en centímetros, solo el número. Si el texto da mm o pulgadas, convierte a cm.",
      "exclusiveMinimum": 0,
      "title": "Ancho Cm",
      "type": "number"
    },
    "material": {
      "description": "Material principal. Si hay varios, el de la superficie más grande. Si no encaja, 'otro'.",
      "enum": [
        "madera",
        "metal",
        "plastico",
        "vidrio",
        "textil",
        "otro"
      ],
      "title": "Material",
      "type": "string"
    },
    "precio_eur": {
      "anyOf"

## 5. Plantilla completa: datos → LLM → objeto validado

Pasos:

1. Defines el modelo Pydantic (ya hecho: `FichaProducto`).
2. Construyes los mensajes: sistema con el esquema + usuario con el texto.
3. Llamas al LLM. **Aquí: mock.**
4. Limpias la salida. Qwen3 puede soltar `<think>…</think>` y bloques de código Markdown.
5. `model_validate_json`. Si falla, le devuelves el error al modelo y reintentas.

### 5.1 Entrada

In [12]:
TEXTO_PROVEEDOR = """Mesa auxiliar NORDA (ref. SKU-3391).
Tablero de roble macizo con patas de acero.
Medidas: 450 mm de ancho x 450 mm de fondo x 520 mm de alto.
Combina con el taburete SKU-1042. PVP 89,00 € IVA incl."""

mensajes = [
    {"role": "system", "content": prompt_de_sistema(FichaProducto)},
    {"role": "user", "content": TEXTO_PROVEEDOR},
]

### 5.2 LLM simulado

Devuelve respuestas enlatadas en orden. La primera está mal a propósito, para ver el reintento. Para usar un modelo de verdad, cambia `llm_falso` por una de las plantillas de 5.5. La firma es la misma: `llm(mensajes, esquema) -> str`.

In [13]:
RESPUESTAS_ENLATADAS = [
    # Intento 1: piensa en voz alta, envuelve en ```json, deja "450 mm" como texto y usa un material
    # fuera de la lista. Falla la validación.
    "<think>El ancho es 450 mm, el material es roble...</think>\n"
    '```json\n{"nombre": "Mesa auxiliar NORDA", "ancho_cm": "450 mm", '
    '"material": "roble", "precio_eur": 89.0}\n```',
    # Intento 2: corrige tras leer los errores.
    '{"nombre": "Mesa auxiliar NORDA", "ancho_cm": 45, "material": "madera", '
    '"precio_eur": 89.0, "compatible_con": ["SKU-1042"]}',
]


def llm_falso(mensajes: list[dict], esquema: dict | None = None) -> str:
    """Mock: la n-ésima llamada de una conversación devuelve la n-ésima respuesta enlatada."""
    n = sum(m["role"] == "assistant" for m in mensajes)
    return RESPUESTAS_ENLATADAS[min(n, len(RESPUESTAS_ENLATADAS) - 1)]

### 5.3 Limpiar la salida

In [14]:
import re


def extraer_json(texto: str) -> str:
    """Quita <think>...</think> y vallas de código. Devuelve de la primera { a la última }."""
    texto = re.sub(r"<think>.*?</think>", "", texto, flags=re.DOTALL)
    inicio, fin = texto.find("{"), texto.rfind("}")
    return texto[inicio : fin + 1] if inicio != -1 and fin > inicio else texto.strip()


print(extraer_json(RESPUESTAS_ENLATADAS[0]))

{"nombre": "Mesa auxiliar NORDA", "ancho_cm": "450 mm", "material": "roble", "precio_eur": 89.0}


### 5.4 Validar y reintentar

Si el JSON no cumple, los errores de Pydantic vuelven al modelo como mensaje de usuario. Son precisos (campo + motivo), así que el modelo suele corregir a la primera.

In [15]:
from collections.abc import Callable
from typing import TypeVar

T = TypeVar("T", bound=BaseModel)


def generar_estructurado(
    modelo: type[T], mensajes: list[dict], llm: Callable[..., str], max_intentos: int = 3
) -> T:
    mensajes = list(mensajes)
    esquema = modelo.model_json_schema()
    for intento in range(1, max_intentos + 1):
        salida = llm(mensajes, esquema)
        try:
            objeto = modelo.model_validate_json(extraer_json(salida))
            print(f"intento {intento}: OK")
            return objeto
        except ValidationError as e:
            errores = e.errors(include_url=False, include_context=False)
            print(f"intento {intento}: MAL -> {[(err['loc'], err['msg']) for err in errores]}")
            mensajes += [
                {"role": "assistant", "content": salida},
                {
                    "role": "user",
                    "content": "Tu JSON no cumple el esquema. Errores: "
                    + json.dumps(errores, ensure_ascii=False, default=str)
                    + ". Devuelve solo el JSON corregido.",
                },
            ]
    raise ValueError(f"sin JSON válido tras {max_intentos} intentos")


ficha = generar_estructurado(FichaProducto, mensajes, llm_falso)
ficha

intento 1: MAL -> [(('ancho_cm',), 'Input should be a valid number, unable to parse string as a number'), (('material',), "Input should be 'madera', 'metal', 'plastico', 'vidrio', 'textil' or 'otro'")]
intento 2: OK


FichaProducto(nombre='Mesa auxiliar NORDA', ancho_cm=45.0, material='madera', precio_eur=89.0, compatible_con=['SKU-1042'])

Ya no es texto. Es un objeto con tipos: se puede operar con él y comparar con la salida esperada.

In [16]:
print(ficha.ancho_cm * 2, "cm si pones dos mesas juntas")

SALIDA_ESPERADA = FichaProducto(
    nombre="Mesa auxiliar NORDA",
    ancho_cm=45,
    material="madera",
    precio_eur=89.0,
    compatible_con=["SKU-1042"],
)
assert ficha == SALIDA_ESPERADA
print("ficha == SALIDA_ESPERADA")
ver(ficha.model_dump())

90.0 cm si pones dos mesas juntas
ficha == SALIDA_ESPERADA
{
  "nombre": "Mesa auxiliar NORDA",
  "ancho_cm": 45.0,
  "material": "madera",
  "precio_eur": 89.0,
  "compatible_con": [
    "SKU-1042"
  ]
}


### 5.5 Plantillas para un LLM real

Esta celda **solo define funciones**. No carga nada ni llama a nadie. Para usarlas, descomenta la última línea.

- **A. Transformers local** (lo que tenéis en la DGX). Nivel 1: prompt + validar + reintentar. Sin dependencias nuevas.
- **B. Servidor compatible con OpenAI** (vLLM, Ollama, OpenAI). Nivel 3: el esquema va en `response_format` y el servidor restringe la decodificación. Requiere `pip install openai`.
  - Con la API oficial de OpenAI, `strict: true` exige `additionalProperties: false` y todos los campos en `required`. El SDK lo adapta solo si usas `client.chat.completions.parse(response_format=FichaProducto)`. Ver [guía de OpenAI](https://developers.openai.com/api/docs/guides/structured-outputs).
  - Con Claude: [structured outputs](https://platform.claude.com/docs/en/build-with-claude/structured-outputs).
  - Con vLLM: [structured outputs](https://docs.vllm.ai/en/latest/features/structured_outputs/). El nombre de los parámetros cambia entre versiones: mira la doc de tu versión.

In [17]:
def hacer_llm_transformers(model_id: str = "Qwen/Qwen3-1.7B", max_new_tokens: int = 512):
    """A. Modelo local con transformers. Devuelve una función llm(mensajes, esquema) -> str."""
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, dtype=torch.bfloat16 if device == "cuda" else torch.float32, device_map=device
    )

    def llm(mensajes: list[dict], esquema: dict | None = None) -> str:
        prompt = tokenizer.apply_chat_template(
            mensajes, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
        entrada = tokenizer(prompt, return_tensors="pt").to(model.device)
        salida = model.generate(**entrada, max_new_tokens=max_new_tokens, do_sample=False)
        return tokenizer.decode(salida[0, entrada.input_ids.shape[1] :], skip_special_tokens=True)

    return llm


def hacer_llm_openai_compatible(
    model: str, base_url: str = "http://localhost:8001/v1", api_key: str = "EMPTY"
):
    """B. Cualquier servidor compatible con OpenAI. El esquema va en response_format (nivel 3)."""
    from openai import OpenAI

    client = OpenAI(base_url=base_url, api_key=api_key)

    def llm(mensajes: list[dict], esquema: dict | None = None) -> str:
        respuesta = client.chat.completions.create(
            model=model,
            messages=mensajes,
            temperature=0,
            response_format={
                "type": "json_schema",
                "json_schema": {"name": "salida", "schema": esquema, "strict": True},
            },
        )
        return respuesta.choices[0].message.content

    return llm


# ficha = generar_estructurado(FichaProducto, mensajes, hacer_llm_transformers())

## 6. Dónde se usa en ARCA

| Fase | Archivo | Qué hace Pydantic ahí |
|---|---|---|
| API | [`api/schemas.py`](../api/schemas.py) | Contrato de entrada y salida de los 4 endpoints. El corrector valida contra estos modelos. |
| 1 | [`rlm/rewards.py`](../rlm/rewards.py), [`rlm/train_grpo.py`](../rlm/train_grpo.py) | Si vuestra tarea tiene salida estructurada, el esquema **es** la recompensa (6.1). |
| 1 | [`rlm/distill.py`](../rlm/distill.py) | Juez LLM que filtra trazas con razonamiento basura: veredicto estructurado (6.3). |
| 2 | [`tool_use/registry.py`](../tool_use/registry.py) | Firma de la función → modelo Pydantic → JSON Schema que ve el modelo. Valida los argumentos que genera (6.2). |
| 3 | [`rag/generate.py`](../rag/generate.py) | Respuesta + citas como JSON, con validación de que las citas existen (6.4). Opcional: el repo usa regex `[id]`. |
| 4 | [`agent/react_agent.py`](../agent/react_agent.py) | Cada paso del agente acaba en `AgentStep` y la respuesta en `AgentResponse` (6.5). |

### 6.1 Fase 1: el esquema como recompensa (GRPO)

Si vuestra tarea es extracción estructurada, la recompensa verificable sale gratis de Pydantic. Misma firma que [`rlm/rewards.py`](../rlm/rewards.py): `reward(prompts, completions, **kwargs) -> list[float]`. El JSON va **dentro** de `<answer>…</answer>`: el formato `<think>` de la fase 1 se mantiene.

Dos recompensas:

- **De esquema:** 1 si cumple, 0.25 si es JSON pero no cumple, 0 si ni es JSON. Sirve para ver cómo sube la tasa de JSON válido durante GRPO.
- **De contenido:** fracción de campos iguales a la salida esperada. Dice qué campos se resisten.

Docs: [TRL `GRPOTrainer`](https://huggingface.co/docs/trl/main/en/grpo_trainer).

In [18]:
from rlm.rewards import _completion_text, extract_answer


def schema_reward(prompts, completions, **kwargs) -> list[float]:
    recompensas = []
    for completion in completions:
        respuesta = extract_answer(_completion_text(completion)) or ""
        try:
            FichaProducto.model_validate_json(respuesta)
            recompensas.append(1.0)
        except ValidationError as e:
            es_json = all(err["type"] != "json_invalid" for err in e.errors())
            recompensas.append(0.25 if es_json else 0.0)
    return recompensas


def field_reward(prompts, completions, answer: list[str], **kwargs) -> list[float]:
    """`answer` es la columna del dataset con la ficha esperada en JSON."""
    recompensas = []
    for completion, esperada_json in zip(completions, answer, strict=True):
        esperada = FichaProducto.model_validate_json(esperada_json).model_dump()
        try:
            predicha = FichaProducto.model_validate_json(
                extract_answer(_completion_text(completion)) or ""
            ).model_dump()
        except ValidationError:
            recompensas.append(0.0)
            continue
        aciertos = sum(predicha[campo] == valor for campo, valor in esperada.items())
        recompensas.append(aciertos / len(esperada))
    return recompensas


completions_simuladas = [
    "<think>450 mm son 45 cm.</think><answer>" + SALIDA_ESPERADA.model_dump_json() + "</answer>",
    '<think>...</think><answer>{"nombre": "Mesa", "ancho_cm": 45, "material": "madera"}</answer>',
    '<think>...</think><answer>{"nombre": "Mesa", "ancho_cm": "450 mm"}</answer>',
    "<think>...</think><answer>la mesa mide 45 cm</answer>",
]
esperadas = [SALIDA_ESPERADA.model_dump_json()] * len(completions_simuladas)

print("schema_reward:", schema_reward(None, completions_simuladas))
print("field_reward: ", [round(r, 2) for r in field_reward(None, completions_simuladas, esperadas)])

schema_reward: [1.0, 1.0, 0.25, 0.0]
field_reward:  [1.0, 0.4, 0.0, 0.0]


### 6.2 Fase 2: una tool call **es** structured output

Cuando el modelo llama a una herramienta genera `{"name": ..., "arguments": {...}}`, y `arguments` tiene que cumplir un esquema. Es el mismo problema de la sección 5.

[`tool_use/registry.py`](../tool_use/registry.py) ya lo resuelve con Pydantic:

- `@tool` lee la firma de la función y crea un modelo Pydantic con `create_model`.
- `tool.definition` es el JSON Schema que ve el modelo. El **docstring** es la `description` de la herramienta: es prompt.
- `registry.call(...)` valida los argumentos generados. Si fallan, el error vuelve al modelo como observación, no revienta el bucle.

**Truco.** El registro solo usa el docstring entero como descripción de la herramienta. La sección `Args:` del docstring **no** llega a cada parámetro. Para que cada argumento lleve su propia `description` (y su `enum`), usa `Annotated[tipo, Field(...)]`. Funciona sin tocar el registro.

Docs: [Qwen function calling](https://qwen.readthedocs.io/en/latest/framework/function_call.html) · [Transformers: tool use en chat templates](https://huggingface.co/docs/transformers/main/en/chat_extras)

In [19]:
from typing import Annotated

from tool_use.registry import ToolRegistry, tool

Unidad = Literal["mm", "cm", "m", "in"]


@tool
def convertir_longitud(
    valor: Annotated[float, Field(gt=0, description="Cantidad a convertir.")],
    desde: Annotated[Unidad, Field(description="Unidad de origen.")],
    hacia: Annotated[Unidad, Field(description="Unidad de destino.")] = "cm",
) -> float:
    """Convierte una longitud entre unidades. Úsala antes de rellenar cualquier campo en cm."""
    a_mm = {"mm": 1, "cm": 10, "m": 1000, "in": 25.4}
    return valor * a_mm[desde] / a_mm[hacia]


ver(convertir_longitud.definition)

{
  "type": "function",
  "function": {
    "name": "convertir_longitud",
    "description": "Convierte una longitud entre unidades. Úsala antes de rellenar cualquier campo en cm.",
    "parameters": {
      "properties": {
        "valor": {
          "description": "Cantidad a convertir.",
          "exclusiveMinimum": 0,
          "title": "Valor",
          "type": "number"
        },
        "desde": {
          "description": "Unidad de origen.",
          "enum": [
            "mm",
            "cm",
            "m",
            "in"
          ],
          "title": "Desde",
          "type": "string"
        },
        "hacia": {
          "default": "cm",
          "description": "Unidad de destino.",
          "enum": [
            "mm",
            "cm",
            "m",
            "in"
          ],
          "title": "Hacia",
          "type": "string"
        }
      },
      "required": [
        "valor",
        "desde"
      ],
      "type": "object"
    }
  }
}


Argumentos que "generó el modelo". El primero cumple. El segundo no: `registry.call` no lanza excepción, devuelve el error para que el modelo lo lea y corrija.

In [20]:
registro = ToolRegistry().register(convertir_longitud)

for argumentos in [{"valor": 450, "desde": "mm"}, {"valor": "mucho", "desde": "pies"}]:
    rec = registro.call("convertir_longitud", argumentos)
    print("ok" if rec.ok else "MAL", "->", rec.result if rec.ok else rec.error, "\n")

ok -> 45.0 

MAL -> invalid arguments: [{'type': 'float_parsing', 'loc': ('valor',), 'msg': 'Input should be a valid number, unable to parse string as a number', 'input': 'mucho'}, {'type': 'literal_error', 'loc': ('desde',), 'msg': "Input should be 'mm', 'cm', 'm' or 'in'", 'input': 'pies', 'ctx': {'expected': "'mm', 'cm', 'm' or 'in'"}}] 



### 6.3 Fase 1: juez LLM con veredicto estructurado

[`rlm/distill.py`](../rlm/distill.py) sugiere un juez LLM para tirar trazas correctas por suerte. El juez devuelve un veredicto que tu código tiene que leer: structured output.

Fíjate en el **orden**: `razonamiento` va primero. El modelo genera en orden, así que primero justifica y luego decide. Al revés decide a ciegas y luego inventa la justificación. Forzar JSON también puede empeorar el razonamiento: [Tam et al., 2024, "Let Me Speak Freely?"](https://arxiv.org/abs/2408.02442).

In [21]:
class VeredictoJuez(BaseModel):
    """Evaluación de una traza de razonamiento cuya respuesta final ya es correcta."""

    razonamiento: str = Field(
        description=(
            "Dos o tres frases: ¿cada paso se sigue del anterior? "
            "¿hay saltos o cálculos sin sentido?"
        )
    )
    razonamiento_valido: bool = Field(
        description=(
            "true solo si el razonamiento justifica la respuesta. false si acierta por suerte."
        )
    )
    paso_erroneo: int | None = Field(
        default=None,
        description="Número del primer paso incorrecto, empezando en 1. null si no hay.",
    )


salida_juez = (
    '{"razonamiento": "El paso 2 divide entre 100 en vez de entre 10, pero el paso 3 vuelve a '
    'multiplicar y el error se cancela.", "razonamiento_valido": false, "paso_erroneo": 2}'
)
veredicto = VeredictoJuez.model_validate_json(salida_juez)
print("quedarse con la traza:", veredicto.razonamiento_valido)
print("primer paso malo:", veredicto.paso_erroneo)
print(list(VeredictoJuez.model_json_schema()["properties"]))  # orden de generación

quedarse con la traza: False
primer paso malo: 2
['razonamiento', 'razonamiento_valido', 'paso_erroneo']


### 6.4 Fase 3: respuesta con citas que existen

[`rag/generate.py`](../rag/generate.py) saca las citas con una regex `[id]`. Alternativa: pedir respuesta y citas como JSON y **validar que cada cita está entre los fragmentos recuperados**. Los ids válidos cambian en cada pregunta, así que se pasan en `context=` y el validador los lee. Una cita inventada se convierte en `ValidationError`: puedes reintentar o marcarla.

Al final se vuelca a `RagResponse`, que es lo que mira el corrector.

In [22]:
from pydantic import ValidationInfo, field_validator

from api.schemas import RagResponse, RetrievedChunk


class RespuestaRAG(BaseModel):
    """Respuesta apoyada solo en los fragmentos recuperados."""

    answer: str = Field(
        description=(
            "Respuesta en español. Tras cada dato, el id del fragmento entre corchetes, "
            "como [doc1-0003]."
        )
    )
    citations: list[str] = Field(
        description=(
            "Ids de los fragmentos usados. Solo ids que aparecen en la información recuperada."
        )
    )

    @field_validator("citations")
    @classmethod
    def citas_existen(cls, citas: list[str], info: ValidationInfo) -> list[str]:
        validos = (info.context or {}).get("ids_validos")
        inventadas = [c for c in citas if validos is not None and c not in validos]
        if inventadas:
            raise ValueError(f"ids que no están en los fragmentos recuperados: {inventadas}")
        return citas


chunks = [
    RetrievedChunk(
        id="norma-0007", source="etiquetado.pdf", score=0.81, text="Las medidas van en cm..."
    ),
    RetrievedChunk(
        id="estilo-0002", source="manual_estilo.md", score=0.64, text="El nombre va sin SKU..."
    ),
]
ids_validos = {c.id for c in chunks}

buena = '{"answer": "Las medidas van en cm [norma-0007].", "citations": ["norma-0007"]}'
inventada = '{"answer": "Va en pulgadas [norma-0099].", "citations": ["norma-0099"]}'

for salida in [buena, inventada]:
    try:
        r = RespuestaRAG.model_validate_json(salida, context={"ids_validos": ids_validos})
        rag = RagResponse(answer=r.answer, chunks=chunks, citations=r.citations, retriever="hybrid")
        print("OK ->", rag.citations)
    except ValidationError as e:
        print("MAL ->", e.errors()[0]["msg"])

OK -> ['norma-0007']
MAL -> Value error, ids que no están en los fragmentos recuperados: ['norma-0099']


### 6.5 Fase 4: del texto del modelo a `AgentStep`

En el agente, Qwen escribe la acción como JSON dentro de `<tool_call>…</tool_call>`. [`tool_use/parser.py`](../tool_use/parser.py) saca el JSON, el registro valida los argumentos con Pydantic y cada vuelta se guarda como `AgentStep`. Al final todo se empaqueta en `AgentResponse`: si falta un campo, falla aquí y no en el corrector.

Bucle mínimo con salidas simuladas (sin memoria ni presupuesto de tokens: eso está en [`agent/react_agent.py`](../agent/react_agent.py)).

In [23]:
from agent.react_agent import final_answer
from api.schemas import AgentAction, AgentResponse, AgentStep
from tool_use.parser import parse_tool_calls

registro.register(final_answer)

SALIDAS_AGENTE = [
    "<think>El proveedor da 450 mm. La ficha va en cm. Convierto.</think>\n"
    '<tool_call>\n{"name": "convertir_longitud", '
    '"arguments": {"valor": 450, "desde": "mm", "hacia": "cm"}}\n</tool_call>',
    "<think>Ya tengo 45 cm.</think>\n"
    '<tool_call>\n{"name": "final_answer", '
    '"arguments": {"answer": "La mesa NORDA mide 45 cm de ancho."}}\n</tool_call>',
]

pasos: list[AgentStep] = []
respuesta_final = ""
for salida in SALIDAS_AGENTE:
    parsed = parse_tool_calls(salida)
    call = parsed.tool_calls[0]
    accion = AgentAction(name=call.name, arguments=call.arguments)
    rec = registro.call(call.name, call.arguments)
    observacion = str(rec.result if rec.ok else rec.error)
    pasos.append(AgentStep(thought=parsed.thinking, action=accion, observation=observacion))
    if call.name == "final_answer":
        respuesta_final = observacion
        break

respuesta = AgentResponse(
    final_answer=respuesta_final, steps=pasos, n_steps=len(pasos), model="mock"
)
ver(respuesta.model_dump())

{
  "final_answer": "La mesa NORDA mide 45 cm de ancho.",
  "steps": [
    {
      "thought": "El proveedor da 450 mm. La ficha va en cm. Convierto.",
      "action": {
        "name": "convertir_longitud",
        "arguments": {
          "valor": 450,
          "desde": "mm",
          "hacia": "cm"
        }
      },
      "observation": "45.0"
    },
    {
      "thought": "Ya tengo 45 cm.",
      "action": {
        "name": "final_answer",
        "arguments": {
          "answer": "La mesa NORDA mide 45 cm de ancho."
        }
      },
      "observation": "La mesa NORDA mide 45 cm de ancho."
    }
  ],
  "n_steps": 2,
  "succeeded": true,
  "model": "mock",
  "tokens_used": 0
}


## 7. Errores típicos

- **Forzar JSON a un modelo que razona.** Baja la calidad del razonamiento. Deja pensar primero (`<think>` o campo `razonamiento` el primero) y extrae después.
- **Esquema enorme o muy anidado.** Más tokens, más fallos. Plano y pequeño.
- **`Optional` sin decir cuándo es `null`.** El modelo rellena con lo que sea. Dilo en la `description`.
- **`str` libre donde cabe `Literal`.** Pierdes el `enum` y la validación.
- **Fiarse de la gramática.** Valida siempre con Pydantic.
- **Campos desconocidos.** Si no quieres que el modelo añada campos, `model_config = ConfigDict(extra="forbid")`. En el esquema sale `additionalProperties: false`.
- **Tocar `api/schemas.py`.** Renombrar o quitar campos rompe el corrector. Añadir opcionales, sí.
- **Tirar los fallos.** Guarda cuántas salidas fallan la validación y por qué campo. Es una métrica para `EXPERIMENTS.md`.

## 8. Links

**Pydantic**
- [Documentación](https://pydantic.dev/docs/validation/latest/get-started/)
- [Modelos](https://pydantic.dev/docs/validation/latest/concepts/models/) · [Campos y `Field`](https://pydantic.dev/docs/validation/latest/concepts/fields/) · [JSON Schema](https://pydantic.dev/docs/validation/latest/concepts/json_schema/) · [Validadores](https://pydantic.dev/docs/validation/latest/concepts/validators/) · [Tipos de error](https://pydantic.dev/docs/validation/latest/errors/validation_errors/)
- [Understanding JSON Schema](https://json-schema.org/understanding-json-schema/reference)

**API**
- [FastAPI: cuerpo de la petición](https://fastapi.tiangolo.com/tutorial/body/) · [`response_model`](https://fastapi.tiangolo.com/tutorial/response-model/) · [Tests](https://fastapi.tiangolo.com/tutorial/testing/)

**Structured Output**
- [OpenAI: Structured Outputs](https://developers.openai.com/api/docs/guides/structured-outputs)
- [Claude: Structured outputs](https://platform.claude.com/docs/en/build-with-claude/structured-outputs)
- [vLLM: Structured outputs](https://docs.vllm.ai/en/latest/features/structured_outputs/)
- [Ollama: Structured outputs](https://ollama.com/blog/structured-outputs)
- [Outlines](https://github.com/dottxt-ai/outlines) · [XGrammar](https://github.com/mlc-ai/xgrammar) · [Instructor](https://python.useinstructor.com/) (Pydantic + reintentos ya hechos)
- Paper: [Willard & Louf, 2023: Efficient Guided Generation for LLMs](https://arxiv.org/abs/2307.09702)
- Paper: [Tam et al., 2024: Let Me Speak Freely? Format restrictions y razonamiento](https://arxiv.org/abs/2408.02442)

**Tool use y entrenamiento**
- [Qwen: function calling](https://qwen.readthedocs.io/en/latest/framework/function_call.html)
- [Transformers: tool use en chat templates](https://huggingface.co/docs/transformers/main/en/chat_extras)
- [TRL: `GRPOTrainer`](https://huggingface.co/docs/trl/main/en/grpo_trainer)

**En este repo**
- [`api/schemas.py`](../api/schemas.py) · [`api/README.md`](../api/README.md)
- [`tool_use/registry.py`](../tool_use/registry.py) · [`tool_use/parser.py`](../tool_use/parser.py)
- [`rlm/rewards.py`](../rlm/rewards.py) · [`rlm/distill.py`](../rlm/distill.py)
- [`rag/generate.py`](../rag/generate.py) · [`agent/react_agent.py`](../agent/react_agent.py)
- [`docs/temas_ejemplo.md`](temas_ejemplo.md) (ficha de producto: el esquema como recompensa)